# Implementing Naïve Bayes

## 1. Implementing Naïve Bayes from scratch

In [9]:
import numpy as np

In [10]:
# Before we develop the model, let's define the toy dataset we just worked with:
X_train = np.array([
	[0, 1, 1],
	[0, 0, 1],
	[0, 0, 0],
	[1, 1, 0]
])
Y_train = ["Y", "N", "Y", "Y"]
X_test = np.array([[1, 1, 0]])

In [11]:
# For the model, starting with the prior, we first group the data by label and record their indices by classes:
def get_label_indices(labels):
	"""
	Group samples based on their labels and return indices
	@param labels: list of labels
	@return: dict, {class1: [indices], class2: [indices]}
	"""
	from collections import defaultdict
	
	label_indices = defaultdict(list)
	
	for index, label in enumerate(labels):
		label_indices[label].append(index)
	
	return label_indices

In [12]:
label_indices = get_label_indices(Y_train)
print("label_indices:\n", label_indices)

label_indices:
 defaultdict(<class 'list'>, {'Y': [0, 2, 3], 'N': [1]})


In [13]:
# With label_indices, we calculate the prior:
def get_prior(label_indices):
	"""
	Compute prior based on training samples
	@param label_indices: grouped sample indices by class
	@return: dictionary, with class label as key, corresponding prior as the value
	"""
	prior = {
		label: len(indices) for label, indices in label_indices.items()
	}
	total_count = sum(prior.values())
	
	for label in prior:
		prior[label] /= total_count
	
	return prior

In [14]:
prior = get_prior(label_indices)
print("Prior:", prior)

Prior: {'Y': 0.75, 'N': 0.25}


In [15]:
# With prior calculated, we continue with likelihood, which is the conditional probability, P(feature|class):
def get_likelihood(features, label_indices, smoothing=0):
	"""
	Compute likelihood based on training samples
	@param features: matrix of features
	@param label_indices: grouped sample indices by class
	@param smoothing: integer, additive smoothing parameter
	@return: dictionary, with class as key, corresponding conditional probability P(feature|class) vector as value
	"""
	likelihood = {}
	
	for label, indices in label_indices.items():
		likelihood[label] = features[indices, :].sum(axis=0) + smoothing
		total_count = len(indices)
		likelihood[label] = likelihood[label] / (total_count + 2 * smoothing)
	
	return likelihood

In [16]:
# Set the smoothing value to 1 here, which can also be 0 for no smoothing, or any other positive value, as long as a higher classification performance is achieved:
smoothing = 1
likelihood = get_likelihood(X_train, label_indices, smoothing)

print("Likelihood:\n", likelihood)

Likelihood:
 {'Y': array([0.4, 0.6, 0.4]), 'N': array([0.33333333, 0.33333333, 0.66666667])}


In [17]:
# With prior and likelihood ready, we can now compute posterior for the testing/new samples:
def get_posterior(X, prior, likelihood):
  """
  Compute posterior of testing samples, based on prior and likelihood
  @param X: testing samples
  @param prior: dictionary, with class label as key, corresponding prior as the value
  @param likelihood: dictionary, with class label as key, corresponding conditional probability vector as value
  @return: dictionary, with class label as key, corresponding posterior as value
  """
  posteriors = []

  for x in X:
    # posterior is proportional to prior * likelihood
    posterior = prior.copy()
    
    for label, likelihood_label in likelihood.items():
      for index, bool_value in enumerate(x):
        posterior[label] *= likelihood_label[index] if bool_value else (1 - likelihood_label[index])
    
    # normalize so that all sums up to 1
    sum_posterior = sum(posterior.values())

    for label in posterior:
      if posterior[label] == float('inf'):
        posterior[label] = 1.0
      else:
        posterior[label] /= sum_posterior
    
    posteriors.append(posterior.copy())

  return posteriors

In [18]:
# Now, let's predict the class of our one sample test set using this prediction function:
posterior = get_posterior(X_test, prior, likelihood)

print("Posterior:\n", posterior)

Posterior:
 [{'Y': np.float64(0.9210360075805433), 'N': np.float64(0.07896399241945673)}]


We have successfully implemented Naïve Bayes from scratch.

## 2. Implementing Naïve Bayes with scikit-learn

>>> 💡 **Coding from scratch and implementing your own solutions is the best way to learn about machine learning models.**

In [19]:
from sklearn.naive_bayes import BernoulliNB

In [20]:
# Initialize a model with a smoothing factor (specified as alpha in scikit-learn) of 1.0, and prior learned from the training set (specified as fit_prior=True in scikit-learn):
clf = BernoulliNB(alpha=1.0, fit_prior=True)

# Train the Naïve Bayes classifier with the fit method:
clf.fit(X_train, Y_train)

# Obtain the predicted probability results with the predicted_proba method:
pred_prob = clf.predict_proba(X_test)

print("[scikit-learn] Predicted probabilities:\n", pred_prob)

[scikit-learn] Predicted probabilities:
 [[0.07896399 0.92103601]]


In [21]:
# Directly acquire the predicted class with the predict method (0.5 is the default threshold, if the predicted probability of class Y is > 0.5, class Y is assigned; otherwise, N is used):
pred = clf.predict(X_test)

print("[scikit-learn] Prediction:", pred)

[scikit-learn] Prediction: ['Y']
